In [1]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt


In [2]:
df = pd.read_csv("laptop_price.csv", encoding='latin-1')
df.head()

In [3]:
df.shape

(1594, 13)

In [4]:
df.info()

In [5]:
df.describe()

In [6]:
df.isnull().sum()

In [7]:
df.duplicated().sum()

In [8]:
df.columns

In [9]:
df['Ram'] = df['Ram'].astype(str).str.replace('GB', '').astype(int)
df['Weight'] = df['Weight'].astype(str).str.replace('kg', '').astype(float)


In [10]:
category = ['Company', 'TypeName', 'ScreenResolution', 'Cpu', 'Memory', 'Gpu', 'OpSys']
df_encoded = pd.get_dummies(df, columns=category, drop_first=True)


In [11]:
df_encoded = df_encoded.drop(columns=['laptop_ID', 'Product'])
df_encoded = df_encoded.astype(int)


In [12]:
from sklearn.preprocessing import StandardScaler
numerical_feature = ['Inches', 'Ram', 'Weight']
scaler = StandardScaler()
df_encoded[numerical_feature] = scaler.fit_transform(df_encoded[numerical_feature])


In [13]:
from sklearn.model_selection import train_test_split
X = df_encoded.drop(columns=['Price_euros'])
y = df_encoded['Price_euros']
x_train, x_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)


In [14]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping

input_dim = x_train.shape[1]
model = Sequential([
    Dense(units=128, activation='relu', input_shape=(input_dim,)),
    Dropout(0.2),
    Dense(units=128, activation='relu'),
    Dropout(0.2),
    Dense(units=64, activation='relu'),
    Dropout(0.2),
    Dense(units=32, activation='relu'),
    Dropout(0.2),
    Dense(units=1, activation='linear')
])

early = EarlyStopping(monitor='val_loss', verbose=1, patience=10, restore_best_weights=True)
model.compile(optimizer='adam', loss='mean_squared_error', metrics=['mean_absolute_error'])
model.summary()


In [15]:
history = model.fit(x_train, y_train, epochs=120, validation_split=0.2, callbacks=[early], batch_size=32)


In [16]:
y_pred = model.predict(x_test)
y_pred_train = model.predict(x_train)


In [17]:
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
mse = mean_squared_error(y_test, y_pred)
mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)
print('MSE:', mse)
print('MAE:', mae)
print('R2 Score:', r2)


MSE: 84120.33
MAE: 201.34
R2 Score: 0.8411


In [18]:
plt.plot(history.history['loss'], label='train_loss')
plt.plot(history.history['val_loss'], label='val_loss')
plt.legend()
plt.show()


In [19]:
import pickle

model.save('laptop_price_model.keras')

model_columns = X.columns.tolist()
with open('model_columns.pkl', 'wb') as f:
    pickle.dump(model_columns, f)

dropdowns = {
    'Company': sorted(df['Company'].unique().tolist()),
    'TypeName': sorted(df['TypeName'].unique().tolist()),
    'Cpu_brand': sorted(df['Cpu'].unique().tolist()),
    'Gpu_brand': sorted(df['Gpu'].unique().tolist()),
    'OpSys': sorted(df['OpSys'].unique().tolist()),
    'Ram': sorted(df['Ram'].unique().tolist()),
    'ScreenResolution': sorted(df['ScreenResolution'].unique().tolist()),
    'Memory': sorted(df['Memory'].unique().tolist())
}

with open('dropdowns.pkl', 'wb') as f:
    pickle.dump(dropdowns, f)

with open('scaler.pkl', 'wb') as f:
    pickle.dump(scaler, f)

print('Files generated: laptop_price_model.keras, model_columns.pkl, dropdowns.pkl, scaler.pkl')


Files generated: laptop_price_model.keras, model_columns.pkl, dropdowns.pkl, scaler.pkl
